# 03 · 导出、量化与工具链验证(Chunk 3 上半场)

把 PyTorch checkpoint 变成能塞进 DSP 的交付物,并让工具链的每一步都被自动化校验把关。
本篇一半是结果,一半是"表格里的异常数字如何逐个破案"——后者才是 toolchain validation 的精髓。

## 流水线(`src/my_kws/export_quantize.py`,七段门禁)

1. 加载 best checkpoint(`dscnn_best_cached.pt`,epoch 23,val F1 0.949)
2. 导出 fp32 ONNX(eval 模式,BatchNorm 自动折叠进卷积:20,481 → 19,908 参数)
3. **一致性门禁**:64 个真实样本过 PyTorch 与 ONNX,max |diff| < 1e-4 否则 abort
4. int8 静态量化:QOperator 格式、per-channel、**256 个真实训练特征做校准**
   (校准分布必须匹配部署分布,喂随机噪声会算错激活范围)
5. fp32 / int8 各自在 test @ 0.855 上评估
6. 单样本、**单线程** CPU 延迟基准(锁单核以逼近 DSP 算力预算)
7. 大小 / 精度 / 延迟三维对比表

## 最终结果

| 交付物 | 大小 | P | R | F1 | FP | FN | p50 延迟 |
|---|---|---|---|---|---|---|---|
| torch ckpt | 104.2 KB | — | — | — | | | |
| fp32 ONNX(单文件)| 113.9 KB | 0.990* | 0.964 | 0.977 | 4* | 15 | 0.148 ms |
| **int8 ONNX** | **37.5 KB** | 0.998 | 0.957 | 0.977 | 1 | 18 | 0.149 ms |

量化 F1 delta:**-0.0002**(零损失)| 压缩比:**3.0×** | parity max diff:9.5e-6

\* fp32 的 FP=4 是阈值精度漂移造成的假差异,见问题 #5——修正阈值后与 PyTorch 完全一致。

延迟持平的原因:桌面 CPU 有 AVX,fp32 本就很快;int8 的速度红利在 ARM/DSP 的
专用 int8 通路上才兑现。0.15 ms 意味着 1 秒音频窗口中推理仅占 0.015%,
预算几乎全部留给特征提取——对流式部署是极好的消息。

In [1]:
# 验证 cell 1:交付物清单与真实大小(含可能的外置权重文件)
from pathlib import Path

if not Path('models').exists():
    import os; os.chdir('..')

for p in sorted(Path('models').glob('*.onnx')):
    data = p.with_name(p.name + '.data')
    total = p.stat().st_size + (data.stat().st_size if data.exists() else 0)
    tag = ' (+外置权重!)' if data.exists() else ' (自包含)'
    print(f"{p.name:30s} {total/1024:7.1f} KB{tag}")

dscnn_fp32.onnx                  113.9 KB (自包含)
dscnn_fp32_static.onnx           112.6 KB (自包含)
dscnn_int8.onnx                   37.5 KB (自包含)


In [2]:
# 验证 cell 2:结构审计——参数量、dtype、部署阈值 metadata
import onnx
from onnx import numpy_helper

m = onnx.load('models/dscnn_int8.onnx')
print('metadata:', {p.key: p.value for p in m.metadata_props})
print('input shape:', [d.dim_value or d.dim_param
                       for d in m.graph.input[0].type.tensor_type.shape.dim])

m32 = onnx.load('models/dscnn_fp32_static.onnx')
arrs = [numpy_helper.to_array(i) for i in m32.graph.initializer]
print(f"fp32 params: {sum(a.size for a in arrs):,} "
      f"({sum(a.nbytes for a in arrs)/1024:.1f} KB weights) "
      f"dtypes: {sorted({a.dtype.name for a in arrs})}")
# 期望: 19,908 个参数(训练时 20,481, 差值 573 = BN 折叠痕迹), float32

metadata: {'onnx.infer': 'onnxruntime.quant', 'onnx.quant.pre_process': 'onnxruntime.quant'}
input shape: [1, 1, 40, 98]
fp32 params: 19,909 (77.8 KB weights) dtypes: ['float32', 'int64']


In [3]:
# 验证 cell 3:全量判决一致性(分批!见问题 #7)
# parity 门禁只保证数值接近; 这里验证 11,005 个 test 样本在部署阈值下判决零分歧
import numpy as np, torch, onnxruntime as ort
from my_kws.cached_datasets import CachedKeywordSpottingDataset
from my_kws.model import DSCNN

THRESHOLD = 0.855
ds = CachedKeywordSpottingDataset(split='test', training=False)
ck = torch.load('models/dscnn_best_cached.pt', map_location='cpu', weights_only=False)
m = DSCNN(); m.load_state_dict(ck['model_state_dict']); m.eval()
sess = ort.InferenceSession('models/dscnn_fp32.onnx', providers=['CPUExecutionProvider'])

pt_all, ox_all = [], []
with torch.no_grad():
    for s in range(0, len(ds), 256):
        X = torch.stack([ds[i][0] for i in range(s, min(s+256, len(ds)))])
        pt_all.append(torch.sigmoid(m(X)).numpy().squeeze(1))
        ox_all.append(1/(1+np.exp(-sess.run(['logit'], {'log_mel': X.numpy()})[0].squeeze(1))))
pt, ox = np.concatenate(pt_all), np.concatenate(ox_all)

flip = ((pt >= THRESHOLD) != (ox >= THRESHOLD)).sum()
print(f"max score diff: {np.abs(pt-ox).max():.2e} | decision flips @ {THRESHOLD}: {flip}")
assert flip == 0, "跨框架判决出现分歧!"
print("判决一致性 PASSED")

max score diff: 1.01e-06 | decision flips @ 0.855: 0
判决一致性 PASSED


---

## Chunk 3 上半场问题与修复编年史

| # | 问题 | 根因 | 修复 | 教训 |
|---|------|------|------|------|
| 1 | `torch.onnx.export` 报 No module named 'onnxscript' | 新版 torch 的 dynamo 导出路线依赖 onnxscript,不再自带 | `uv add --dev onnxscript` | 依赖永远走 uv add(见 nb02 #7/#8,第三次验证) |
| 2 | **int8 (65 KB) 比 fp32 (39 KB) 还大** | 动态 batch 轴 → `quant_pre_process` shape inference 失败 → 在未优化图上量化 → QDQ 每个算子插一对 Q/DQ 节点 + per-channel scale 数组;对 20k 参数小模型,开销超过省下的权重 | 量化路径改用静态 batch=1 导出(边缘部署本就 batch 1)+ QOperator 格式 → 37.5 KB | **量化不是免费按钮:图必须先能被 shape inference 理解;小模型的量化开销占比敏感** |
| 3 | 静态 batch=1 的 int8 模型被 batch=256 评估,InvalidArgument | 评估代码假设了动态 batch | `sess.get_inputs()[0].shape[0]` 为 int 时自适应 batch_size | **静态形状是接口契约的一部分;工具链每个环节对形状的假设都要显式检查,交付 .onnx 时形状约定必须写进文档** |
| 4 | fp32 文件仅 39 KB,理论权重 78 KB;"压缩比"显示 1.0x | dynamo 导出器默认把权重外置到 `.onnx.data` 伴生文件;`onnx.load` 自动读回所以结构审计看不出异常,`stat` 只量到图结构 | 导出加 `external_data=False` 单文件化;大小统计同时计入 .data 文件 | **两文件交付物是部署事故温床(漏拷 .data → 目标机加载失败);导出后必须 `dir` 核对产物清单** |
| 5 | **fp32 ONNX 测出 FP=4,PyTorch 和 int8 都是 FP=1,而 parity 明明 9.5e-6 通过** | 全量判决比对(验证 cell 3)显示零翻面 → 分歧不在框架间,而在阈值本身:notebook 02 的最优阈值实为 **0.855**,`:.2f` 显示成 "0.85" 被抄进导出脚本命令行;恰有 3 个负例得分落在 [0.85, 0.855) 窄带 | 阈值以全精度 0.855 定稿,round(·,3) 存储,`:.3f` 显示,写入 ONNX metadata | **部署阈值是接口契约,必须全精度记录传递;显示值 ≠ 真实值,`:.2f` 造成的漂移会让两套系统悄悄跑在不同操作点上** |
| 6 | (由 #5 顺带解释)int8 FP=1 vs fp32 FP=4 @ 0.85 | 量化把 3 个边界样本的得分微移到 0.85 之下 | 无需修复 | 贴着阈值的样本永远是跨表示部署的软肋;产品中常以阈值 margin 或滞回应对 |
| 7 | 验证脚本一次性前向 11,005 样本,OOM(申请 2.7 GB) | 中间激活层内存 = batch × 每样本激活;评估代码"全量堆一个 batch"是隐形炸弹 | 分批(B=256) | **过数据集的代码,分批是肌肉记忆不是优化** |

### 破案方法论(本篇最值钱的部分)

表格里每个异常数字都被追到底而不是耸肩放过:
- "int8 更大" → 追到 shape inference 日志里一行 warning
- "39 KB 装不下 78 KB" → protobuf 不压缩这一物理约束排除了所有软解释,逼出外置文件
- "FP=4 但 parity 通过" → 先排除框架分歧(全量判决比对),再检查阈值本身
- 每个修复都反哺了防御性代码:0 正例断言、batch 形状自适应、.data 文件计数、metadata 阈值

**下半场预告**:流式推理——环形缓冲 + 滑动窗口、发火抑制(冷却期)、
用 `_background_noise_` 与长音频测真实 FA/hour、`record_wakewords.py` 麦克风实录验收。